In [1]:
from edgar import set_identity, Company, get_filings
from edgar.entity import EntityFiling, EntityFilings
from ticker_enum import Ticker as T
from edgar import get_current_filings
from pprint import pprint
import json


set_identity("Loy Yeeko koloyyee@gmail.com")

/Users/loyyeeko/Code/Personal/Finance/sec-daily/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
company = Company("HOOD")
form4s = [f.obj() for f in company.get_filings(form=144).latest(10)]


In [4]:
from edgar import get_filings

filings = get_filings(form=4, year=2026, quarter=[1,2] )
for f in filings[:5]:
    try:
        form4 = f.obj()
        if form4:
            summary = form4.get_ownership_summary()
            print(summary)
            if "Purchase" in summary.primary_activity and summary.net_change > 10000:
                print(f"{summary.insider_name} bought {summary.net_change:,} shares of {summary.issuer}")
            #elif "Sale" in summary.primary_activity and  summary.net_change < -10000:
            #    print(f"{summary.insider_name} sold {summary.net_change:,} shares of {summary.issuer}")
    except:
        print("something wrong.")

TransactionSummary(reporting_date='2026-06-04', issuer_name='10x Genomics, Inc.', issuer_ticker='TXG', insider_name='Sridhar Kosaraju', position='Director', form_type='4', remarks='Exhibit 24.1 - Power of Attorney', transactions=[TransactionActivity(transaction_type='award', code='A', shares=8829, value=0, price_per_share=0, description='', security_type='non-derivative', security_title='Class A Common Stock', underlying_security='', exercise_date=None, expiration_date=None, footnote_ids='', footnotes_text='')], remaining_shares=np.int64(65278), has_derivative_transactions=False)
TransactionSummary(reporting_date='2026-06-04', issuer_name='10x Genomics, Inc.', issuer_ticker='TXG', insider_name='Alan Mateo', position='Director', form_type='4', remarks='Exhibit 24.1 - Power of Attorney', transactions=[TransactionActivity(transaction_type='award', code='A', shares=8829, value=0, price_per_share=0, description='', security_type='non-derivative', security_title='Class A Common Stock', under

In [ ]:
import feedparser

form4_rss = "https://www.sec.gov/cgi-bin/browse-edgar?action=getcurrent&CIK=&type=4&company=&dateb=&owner=only&start=0&count=100&output=atom"
feedparser.USER_AGENT = "Your Name your@email.com"

In [123]:
import re
feed = feedparser.parse(form4_rss)

#for entry in feed.entries:
    #print(entry.title, entry.link, entry.updated)

def filing_info(entry) :
    date_pattern = r"\b[0-9]{4}-[0-9]{2}-[0-9]{2}\b"
    acc_no_pattern = r"\b[0-9]{10}-[0-9]{2}-[0-9]{6}\b"
    date = re.findall(date_pattern, entry.summary)[0] if re.findall(date_pattern, entry.summary) else "" 
    acc_no = re.findall(acc_no_pattern, entry.summary)[0] if re.findall(acc_no_pattern, entry.summary) else ""
    title = entry.title
    opening = title.find("(")
    closing = title.find(")")
    cik = title[opening + 1:closing]
    return {"date": date, "cik": int(cik), "acc_no": acc_no, "title": title }

filing_basis = list(filter( lambda f: "Reporting" not in f["title"],map( filing_info, feed.entries) ))
print(filing_basis)

[{'date': '2026-06-09', 'cik': 101199, 'acc_no': '0001023710-26-000003', 'title': '4 - UNITED FIRE GROUP INC (0000101199) (Issuer)'}, {'date': '2026-06-09', 'cik': 1866364, 'acc_no': '0001977186-26-000002', 'title': '4 - Webull Corp (0001866364) (Issuer)'}, {'date': '2026-06-09', 'cik': 1585364, 'acc_no': '0001585364-26-000089', 'title': '4 - PERRIGO Co plc (0001585364) (Issuer)'}, {'date': '2026-06-09', 'cik': 1006045, 'acc_no': '0001193125-26-263083', 'title': '4 - IRIDEX CORP (0001006045) (Issuer)'}, {'date': '2026-06-09', 'cik': 1505732, 'acc_no': '0001505732-26-000091', 'title': '4 - Bankwell Financial Group, Inc. (0001505732) (Issuer)'}, {'date': '2026-06-09', 'cik': 1505732, 'acc_no': '0001505732-26-000090', 'title': '4 - Bankwell Financial Group, Inc. (0001505732) (Issuer)'}, {'date': '2026-06-09', 'cik': 1275477, 'acc_no': '0001437749-26-019937', 'title': '4/A - BIMINI CAPITAL MANAGEMENT, INC. (0001275477) (Issuer)'}, {'date': '2026-06-09', 'cik': 1505732, 'acc_no': '000150573

In [126]:
from edgar import Filing

def get_summary(comp_entry):
    filing = Filing(company=str(comp_entry["cik"]), cik=comp_entry["cik"], form="4", 
                    filing_date=comp_entry["date"], accession_no=comp_entry["acc_no"])
    f4 = filing.obj()
    #print(f4.insider_name)
    ## The meat!
    #print(f4.get_ownership_summary())
    #print(filing.filing_url)
    return {"transaction_summary": f4.get_ownership_summary(), "filing_url": filing.filing_url}
    
summaries = list(map(get_summary, filing_basis))
print(summaries)
    
    #company = filing.get_entity()
    #f4s = company.get_filings(form="4")
    #for f4 in f4s:
    #    print(f4.primary_documents)
## or more practically, search by CIK
#from edgar import get_filings

#filings = get_filings(form="4", cik=1530721)
#for filing in filings:
#    print(filing)


[{'transaction_summary': TransactionSummary(reporting_date='2026-06-05', issuer_name='UNITED FIRE GROUP INC', issuer_ticker='UFCS', insider_name='George D Milligan', position='Director', form_type='4', remarks='', transactions=[TransactionActivity(transaction_type='purchase', code='P', shares=4500.0, value=203535.0, price_per_share=45.23, description='', security_type='non-derivative', security_title='Common Stock', underlying_security='', exercise_date=None, expiration_date=None, footnote_ids='', footnotes_text='')], remaining_shares=np.float64(90033.5393), has_derivative_transactions=False), 'filing_url': 'https://www.sec.gov/Archives/edgar/data/101199/000102371026000003/primarydocument.xml'}, {'transaction_summary': TransactionSummary(reporting_date='2026-06-09', issuer_name='Webull Corp', issuer_ticker='BULL', insider_name='Walter A. Bishop', position='Director', form_type='4', remarks='', transactions=[TransactionActivity(transaction_type='exercise', code='M', shares=12500, value=

In [111]:
from edgar import get_all_current_filings, get_filings
from edgar.enums import FormType

#filings = get_filings(form="4", filing_date="2026-06-08")
filings = get_current_filings(form="4", owner = 'include', )
#filings = get_all_current_filings(form="4" )

for filing in filings:
    f4 = filing.obj()
    print(type(f4) is FormType.PROSPECTUS_424B1)
    if type(f4) is FormType.PROSPECTUS_424B1: continue

    #print(f4.insider_name)
    ## The meat!
    #print(f4.get_ownership_summary())
    #print(filing.filing_url)
    

False
False
False
False
False
False
False


No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 13013', cik=2121128, accession_no='0001445546-26-004274')
No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 12999', cik=2119999, accession_no='0001445546-26-004273')
No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 12972', cik=2112775, accession_no='0001445546-26-004272')


False
False
False
False
False
False
False


No XBRL attachments found in filing Filing(form='497J', filing_date='2026-06-09', company='FT 12971', cik=2112774, accession_no='0001445546-26-004271')


False
False


No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066562')
No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066562')
No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066561')
No XBRL attachments found in filing Filing(form='485BXT', filing_date='2026-06-09', company='Investment Managers Series Trust II', cik=1587982, accession_no='0001213900-26-066561')


False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


No XBRL attachments found in filing Filing(form='425', filing_date='2026-06-09', company='Axiom Intelligence Acquisition Corp 1', cik=2057030, accession_no='0001213900-26-066554')


False
False
False
False
False
False
False
False
False
